# Averaging sweep from the original 63.9% setup

**CPU → Run all**, using Zyion for Drive. Reuses the already saved `zuco_averaging_sweep/features` cache. **No EEG loading, extraction, cleaning, new labels or new feature cache.** New results in a separate folder, capped at **15 MB**. The first sweep must have prepared its feature cache.

**ZuCo SR only.** Original complete matched agent-labelled paper subset; cleaned EEG; 105 channel means over224–304ms; positive/negative logistic regression C1, balanced class weights, training-only imputation/scaling; three held-out-reader folds.

The earlier sweep used the same type of EEG/features/model, but **prefiltered readers to≥100trials/class and used different grouping seeds**. This version restores the original cohort, reader order, three folds and grouping/model seeds. It first reruns the original40-trial case and compares its scores, confusion matrix, fold example counts and original-event membership hashes with the historical saved result. A mismatch stops the analysis rather than claiming reproduction.

Then test exact average sizes **1,10,20,...100** with five regroupings (single-trial once). The orange diamond marks the **verified historical40-trial grouping**. The blue line averages five regroupings; its40-trial point need not equal63.9%. Shading is the minimum/maximum across correlated regroupings, not a confidence interval or significance test.

**Difference from the first sweep:** keep the full original cohort in the fixed splits. At larger averaging sizes, a reader/class with too few trials produces no complete average. The table reports contributing readers and test-example counts; this is a coverage trade-off rather than a matched-cohort curve. Leftovers are discarded. No favorable readers/seeds are selected.

Words/sentences may overlap across train/test because only readers are held out. Form averages separately within reader/class/partition, using known test labels. These are class-average separability scores, not single-word prediction. This is exploratory follow-up on a previously examined dataset; full-recording nuisance cleaning remains a limitation.


In [ ]:
# Runtime check: run before installation, Drive access, or analysis.
import os
import subprocess

REQUIRED_RUNTIME = 'CPU'
if os.environ.get('COLAB_TPU_ADDR') or os.environ.get('TPU_NAME'):
    raise RuntimeError(f'This notebook needs {REQUIRED_RUNTIME}, not TPU. Choose Runtime > Change runtime type, then rerun all.')
try:
    _gpu_probe = subprocess.run(
        ['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
        capture_output=True, text=True, timeout=15, check=False)
except FileNotFoundError:
    _gpu_names = ''
except subprocess.TimeoutExpired:
    raise RuntimeError('GPU detection timed out. Reconnect the runtime and rerun this cell.') from None
else:
    if _gpu_probe.returncode:
        raise RuntimeError('GPU detection failed. Reconnect the runtime and rerun this cell.')
    _gpu_names = _gpu_probe.stdout.strip()
if REQUIRED_RUNTIME == 'GPU' and not _gpu_names:
    raise RuntimeError('GPU required. Choose Runtime > Change runtime type > T4 GPU, then Run all. No setup or analysis has started.')
if REQUIRED_RUNTIME == 'CPU' and _gpu_names:
    raise RuntimeError('CPU notebook: choose Runtime > Change runtime type > CPU (or hardware accelerator None), then Run all to avoid using GPU quota.')
print('Runtime verified:', _gpu_names if _gpu_names else 'CPU', flush=True)


## 1. Setup and storage check

In [ ]:
from google.colab import drive, auth
from pathlib import Path
import os, sys, subprocess, json
from IPython.display import display, Markdown, Image
import pandas as pd

drive.mount('/content/drive')
THESIS=Path('/content/drive/MyDrive/Thesis')
OUT=THESIS/'Results/word_sentiment_eeg/zuco_original_cohort_sweep'
REPO=Path('/content/word-sentiment-eeg-original-cohort-sweep')
CODE_REVISION='bcbec5c6f8ec61a212dcaf571cca9f9046485ce9'
def git(*args):
    result=subprocess.run(['git','-c','credential.helper=','-c','http.extraheader=',*map(str,args)],
                          text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                          env=dict(os.environ,GIT_TERMINAL_PROMPT='0'))
    if result.returncode:
        raise RuntimeError('Git setup failed:\n'+result.stdout)
    if result.stdout:print(result.stdout,end='',flush=True)
if not REPO.exists():
    git('clone','--depth','1','https://github.com/parmisbathayan/word-sentiment-eeg.git',REPO)
else:
    if subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True).strip():
        raise RuntimeError('Temporary checkout contains edits. Preserve them before updating.')
if subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()!=CODE_REVISION:
    git('-C',REPO,'fetch','--depth','1','origin',CODE_REVISION)
    fetched=subprocess.check_output(['git','-C',str(REPO),'rev-parse','FETCH_HEAD^{commit}'],text=True).strip()
    if fetched!=CODE_REVISION:raise RuntimeError('Fetched code differs from the pinned commit.')
    git('-C',REPO,'checkout','--detach','FETCH_HEAD')
if subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()!=CODE_REVISION:
    raise RuntimeError('Code checkout verification failed.')
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(REPO/'requirements-paper.txt')],check=True)
print('Code revision:',subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip())

# Verify account quota before loading any EEG. Use Zyion for both prompts.
auth.authenticate_user()
import google.auth
from google.auth.transport.requests import AuthorizedSession
credentials,_=google.auth.default()
response=AuthorizedSession(credentials).get('https://www.googleapis.com/drive/v3/about',
    params={'fields':'storageQuota,user(emailAddress)'},timeout=30)
response.raise_for_status();about=response.json();quota=about['storageQuota']
email=about.get('user',{}).get('emailAddress','unavailable')
print('Authenticated quota account:',email)
if email!='zyionygvmutch39@gmail.com':
    raise RuntimeError('Choose your Zyion account for Drive authentication and mounting; it contains the saved trials.')
if 'limit' not in quota:
    raise RuntimeError('Drive did not expose a storage limit; cannot verify free space.')
FREE_DRIVE_BYTES=max(0,int(quota['limit'])-int(quota.get('usage',0)))
used=sum(p.stat().st_size for p in OUT.rglob('*') if p.is_file()) if OUT.exists() else 0
if FREE_DRIVE_BYTES<max(0,15_000_000-used):
    raise RuntimeError('Reserve up to 15 MB for the feature cache and results. No EEG loading has started.')
print(f'Free space: {FREE_DRIVE_BYTES/1e9:.2f} GB; new output cap: 15 MB.')
print('New output folder:',OUT)


## 2. Verify the historical case, then run the curve

In [ ]:
sys.path.insert(0,str(REPO/'scripts'))
from colab_progress import run_with_progress
command=[sys.executable,'-u',str(REPO/'scripts/zuco_original_cohort_sweep.py'),
         '--thesis-root',str(THESIS),'--free-drive-bytes',str(FREE_DRIVE_BYTES)]
run_with_progress(command,OUT/'runtime_status.json',activity='Original-cohort averaging sweep with historical reference verification')


## 3. Original result verification

In [ ]:
reference=json.loads((OUT/'reference_verification.json').read_text())
print('Historical comparison:',reference['status'])
print(f"Original 40-trial grouping: {reference['balanced_accuracy']*100:.2f}% balanced accuracy")
print('Original test averages:',reference['test_examples'])
print(reference['note'])


## 4. Averaging curve and contributing-reader coverage

In [ ]:
display(Image(filename=str(OUT/'averaging_curve.png')))
scores=pd.read_csv(OUT/'scores.csv')
view=scores.copy()
for c in ['balanced_accuracy','minimum','maximum','dummy']:
    view[c]=(view[c]*100).round(2)
display(view)
cohort=json.loads((OUT/'cohort.json').read_text())
display(pd.DataFrame(cohort['counts']).T.rename_axis('reader'))
print('Original readers:',len(cohort['readers']),'| Original EEG instances:',cohort['events'])
print('Check test_readers at each size: some readers can disappear when no complete group is possible.')
print('Saved results:',OUT)
